# Membership v3 생성

원본(15개) + 파생변수(28개) = **43개 컬럼**

| 구분 | 변수 수 | 변수 |
|------|---------|------|
| 날짜 | 5 | duration_days, reg_weekday, is_weekend_reg, is_month_start_reg, reg_hour_group |
| 가격 | 3 | is_usd, price_per_day, price_tier |
| 상품/기기/결제 | 4 | is_new_product, is_family_plan, device_group, billing_group |
| 인구통계 | 5 | age_group, gender_enc, age_x_screen, is_senior_unverified, is_young_unverified |
| 장르 | 11 | genre_diversity, korean_ratio, avg_showtime, genre_*_ratio x 8 |

**출력**: `data/02_interim/260506_feature_engineering/promotion_0_membership_v3.csv`

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

BASE     = Path('..').resolve()
SPLIT    = BASE / 'data/02_interim/260504_promotion_split'
DATA_OUT = BASE / 'data/02_interim/260506_feature_engineering'
DATA_OUT.mkdir(exist_ok=True)
print('설정 완료')

설정 완료


In [2]:
# 데이터 로드
df = pd.read_csv(SPLIT / 'promotion_0_membership_v2.csv', encoding='utf-8-sig')

um = pd.concat([
    pd.read_csv(SPLIT / 'promotion_0_user_mapping_v2.csv', encoding='utf-8-sig'),
    pd.read_csv(SPLIT / 'promotion_1_user_mapping_v2.csv', encoding='utf-8-sig'),
], ignore_index=True).drop_duplicates()

vh = pd.read_csv(SPLIT / 'promotion_0_view_history_v2.csv', encoding='utf-8-sig').drop_duplicates()

movie = pd.read_csv(BASE / 'data/02_interim/Movie/movie_5171.csv', encoding='utf-8-sig')

print(f'membership:   {df.shape}')
print(f'view_history: {vh.shape}')
print(f'movie:        {movie.shape}')

membership:   (7323, 15)
view_history: (50858, 5)
movie:        (5171, 5)


## 1. 날짜/시간 기반 파생변수

In [3]:
df['reg_date'] = pd.to_datetime(df['reg_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

df['duration_days']      = (df['end_date'] - df['reg_date']).dt.days
df['reg_weekday']        = df['reg_date'].dt.dayofweek
df['is_weekend_reg']     = df['reg_weekday'].isin([5, 6]).astype(int)
df['is_month_start_reg'] = (df['reg_date'].dt.day <= 10).astype(int)
df['reg_hour_group']     = pd.cut(
    df['reg_hour'], bins=[-1, 5, 11, 17, 23], labels=[0, 1, 2, 3]
).astype(float).astype('Int64')

print('날짜 파생변수 완료:', ['duration_days','reg_weekday','is_weekend_reg','is_month_start_reg','reg_hour_group'])

날짜 파생변수 완료: ['duration_days', 'reg_weekday', 'is_weekend_reg', 'is_month_start_reg', 'reg_hour_group']


## 2. 가격 기반 파생변수

In [4]:
df['is_usd']       = (df['price'] < 100).astype(int)
df['price_per_day']= (df['price'] / df['duration_days'].replace(0, np.nan)).round(2)
df['price_tier']   = pd.cut(
    df['price'],
    bins=[0, 100, 5000, 9000, 12000, 99999],
    labels=['달러', '저가', '중가', '고가', '프리미엄']
)

print('가격 파생변수 완료:', ['is_usd','price_per_day','price_tier'])
print(df['price_tier'].value_counts())

가격 파생변수 완료: ['is_usd', 'price_per_day', 'price_tier']
price_tier
중가      2911
달러      2480
고가      1294
프리미엄     529
저가       109
Name: count, dtype: int64


## 3. 상품/기기/결제 기반 파생변수

In [5]:
# 신규 상품 여부 (pk_2xxx = 신규)
df['is_new_product'] = (
    df['product_code'].str.extract(r'pk_(\d+)').astype(float) >= 2000
).astype(int).values.flatten()

# 공유 계정 여부
df['is_family_plan'] = (df['max_screen'] > 1).astype(int)

# 기기 그룹
df['device_group'] = df['payment_device'].map({
    'ios': 'mobile', 'android': 'mobile', 'mobile': 'mobile',
    'pc': 'pc', 'smarttv': 'tv', 'ott': 'tv',
})

# 결제 방식 그룹
df['billing_group'] = df['billing_method'].map({
    121: 'card', 122: 'card', 123: 'card', 129: 'card',
    131: 'mobile_pay', 132: 'mobile_pay',
    134: 'google', 139: 'google',
    140: 'onestore', 141: 'onestore', 142: 'onestore',
    151: 'apple', 170: 'paypal', 180: 'paypal', 190: 'other',
}).fillna('other')

print('상품/기기/결제 파생변수 완료')
print('device_group:', df['device_group'].value_counts().to_dict())
print('billing_group:', df['billing_group'].value_counts().to_dict())

상품/기기/결제 파생변수 완료
device_group: {'mobile': 6118, 'pc': 985, 'tv': 220}
billing_group: {'onestore': 2480, 'google': 1692, 'mobile_pay': 1318, 'apple': 955, 'paypal': 537, 'other': 340, 'card': 1}


## 4. 인구통계 기반 파생변수

In [6]:
df['age_group'] = pd.cut(
    df['age'], bins=[0, 19, 29, 39, 49, 120],
    labels=[0, 1, 2, 3, 4]  # 10/20/30/40/50대+
).astype(float).astype('Int64')

df['gender_enc']           = df['gender'].map({'M': 1, 'F': 0}).fillna(2).astype(int)
df['age_x_screen']         = df['age'] * df['max_screen']
df['is_senior_unverified'] = ((df['age'] >= 50) & (df['is_user_verified'] == 0)).astype(int)
df['is_young_unverified']  = ((df['age'] <= 25) & (df['is_user_verified'] == 0)).astype(int)

print('인구통계 파생변수 완료')
print('age_group 분포:', df['age_group'].value_counts().sort_index().to_dict())

인구통계 파생변수 완료
age_group 분포: {0: 76, 1: 1549, 2: 1513, 3: 3764, 4: 421}


## 5. 장르 기반 파생변수 (View History + Movie)

In [7]:
# showTM → 분 변환
def parse_showtime(s):
    if pd.isna(s): return np.nan
    h = re.search(r'(\d+)시간', str(s))
    m = re.search(r'(\d+)분',   str(s))
    hours, mins = (int(h.group(1)) if h else 0), (int(m.group(1)) if m else 0)
    total = hours * 60 + mins
    return total if total > 0 else np.nan

movie['showtime_min'] = movie['showTM'].apply(parse_showtime)

# VIEW HISTORY에 USER_KEY + 영화 정보 붙이기
vh_key   = vh.merge(um, on='USER_NUM', how='left')
vh_movie = vh_key.merge(
    movie[['MOVIE_ID', 'genre', 'country', 'showtime_min']],
    left_on='MOVIE_NUM', right_on='MOVIE_ID', how='left'
)
print(f'시청이력+영화: {vh_movie.shape}, 장르 결측: {vh_movie["genre"].isna().mean()*100:.1f}%')

시청이력+영화: (50858, 10), 장르 결측: 1.8%


In [8]:
TARGET_GENRES = ['액션', '드라마', '로맨스', '스릴러', '애니메이션', '공포', '코미디', 'SF']

rows = []
for user_key, grp in vh_movie.groupby('USER_KEY'):
    row = {'USER_KEY': user_key}

    # 장르 다양성
    all_genres = set()
    for g in grp['genre'].dropna():
        all_genres.update(x.strip() for x in g.split(','))
    row['genre_diversity'] = len(all_genres)

    # 한국 영화 비율
    total = grp['country'].notna().sum()
    row['korean_ratio'] = grp['country'].str.contains('한국', na=False).sum() / total if total else 0

    # 평균 러닝타임
    row['avg_showtime'] = grp['showtime_min'].mean()

    # 장르별 비율
    genre_total = grp['genre'].notna().sum()
    for g in TARGET_GENRES:
        row[f'genre_{g}_ratio'] = (
            grp['genre'].dropna().str.contains(g).sum() / genre_total if genre_total else 0
        )
    rows.append(row)

genre_df = pd.DataFrame(rows)
print(f'장르 파생변수 생성 완료: {genre_df.shape}')
genre_df.head(3)

장르 파생변수 생성 완료: (7140, 12)


,USER_KEY,genre_diversity,korean_ratio,avg_showtime,genre_액션_ratio,genre_드라마_ratio,genre_로맨스_ratio,genre_스릴러_ratio,genre_애니메이션_ratio,genre_공포_ratio,genre_코미디_ratio,genre_SF_ratio
0,0006075c3c18078eb09940cd27c6359a96a2a17fce8055...,7,0.0,87.000000,0.200000,0.6,0.000000,0.000000,0.6,0.000000,0.400000,0.000000
1,0019ebcf13ea62a20b0e6626103f4d2164e61c64355b9d...,9,0.0,111.500000,0.166667,0.5,0.666667,0.333333,0.0,0.000000,0.666667,0.166667
2,001e7cb9cd0658e839caf6f36441c895f4dede5c53a4a8...,9,0.0,120.636364,0.222222,1.0,0.222222,0.444444,0.0,0.111111,0.111111,0.000000


## 6. 합치고 저장

In [9]:
result = df.merge(genre_df, on='USER_KEY', how='left')

# 시청 이력 없는 유저 장르 피처 → 0
genre_cols = ['genre_diversity', 'korean_ratio', 'avg_showtime'] + \
             [f'genre_{g}_ratio' for g in TARGET_GENRES]
result[genre_cols] = result[genre_cols].fillna(0)

# 저장
out_path = DATA_OUT / 'promotion_0_membership_v3.csv'
result.to_csv(out_path, index=False, encoding='utf-8-sig')

new_features = [
    'duration_days', 'reg_weekday', 'is_weekend_reg', 'is_month_start_reg', 'reg_hour_group',
    'is_usd', 'price_per_day', 'price_tier',
    'is_new_product', 'is_family_plan', 'device_group', 'billing_group',
    'age_group', 'gender_enc', 'age_x_screen', 'is_senior_unverified', 'is_young_unverified',
] + genre_cols

print(f'저장 완료: {out_path}')
print(f'원본 컬럼: 15개  |  추가 파생변수: {len(new_features)}개  |  최종: {result.shape[1]}개')
print(f'최종 shape: {result.shape}')
result.head(3)

저장 완료: C:\Users\USER\OneDrive\바탕 화면\AX git\ott-churn-prediction\kwon.donggeun\data\02_interim\260506_feature_engineering\promotion_0_membership_v3.csv
원본 컬럼: 15개  |  추가 파생변수: 28개  |  최종: 43개
최종 shape: (7323, 43)


,USER_KEY,product_code,price,billing_method,max_screen,is_promotion,is_churn_prevented,is_repurchase,payment_device,is_user_verified,...,korean_ratio,avg_showtime,genre_액션_ratio,genre_드라마_ratio,genre_로맨스_ratio,genre_스릴러_ratio,genre_애니메이션_ratio,genre_공포_ratio,genre_코미디_ratio,genre_SF_ratio
0,3c1f896f1bba7572b29d388a07e01e86fb0d3fbfe51608...,pk_1506,13.49,140,2,0,0,1,ios,0,...,0.0,119.500000,1.000000,0.500000,0.500000,0.000000,0.0,0.000000,0.500000,0.5
1,21026cbc660d69f02985f06f89311143560d14f56da617...,pk_2026,10900.00,151,2,0,0,1,android,1,...,0.0,113.000000,0.666667,0.666667,0.000000,0.666667,0.0,0.000000,0.000000,0.0
2,2c9e3f9186e5dcd624c026c3a72b3070a03822476c6d1c...,pk_1487,7900.00,134,1,0,0,0,android,1,...,0.0,128.722222,0.166667,0.722222,0.166667,0.055556,0.0,0.055556,0.055556,0.0
